# Dependency Management with uv — Bob Learns New Tricks

*One reformed admin, one project, three files: `pyproject.toml`, `uv.lock`, `.venv`.*

## Prologue — Bob, Reformed

> **Bob:** Fifteen years of "one machine, one Python", and Alice knocked it down in an afternoon. Fine. Virtual environments per project — I put it in the runbook myself.
>
> **Bob:** But she left a sticky note on my monitor: *"Next time, don't manage the environments by hand. Look at uv."* One tool that declares, locks, and installs dependencies? That I have to see.

This notebook is Bob working through `uv` dependency management on his own, one command at a time. Three files carry the whole story:

| File | Role |
| --- | --- |
| `pyproject.toml` | **Intent** — which packages, which version ranges |
| `uv.lock` | **Truth** — the fully resolved graph with exact versions |
| `.venv` | **Reality** — what is actually installed |

## Act I — A Clean Desk

### What Is on the Machine?

> **Bob:** First rule of any runbook: know your starting point.

The dev container ships `uv` preinstalled. Verify the version:

In [ ]:
%%bash
set -euxo pipefail
uv --version

### Remove Old Experiments

Remove any project files left over from a previous run, so every command below starts from a known state:

In [ ]:
%%bash
set -euxo pipefail
rm -rf pyproject.toml uv.lock .venv

## Act II — Declaring Intent

### Bob Writes a `pyproject.toml`

> **Bob:** Alright. No `pip install` into anything. First, I write down what the project *needs*.

The `pyproject.toml` declares intent, not exact installs. `httpx` is exact-pinned on purpose — the upgrade story later needs it. Create the file:

In [ ]:
%%bash
set -euxo pipefail
cat > pyproject.toml <<'EOF'
[project]
name = "bobs-license-service"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "fastapi>=0.115,<0.116",
    "httpx==0.24.1",
    "rich>=13.7,<14",
]
EOF

Read the file back before doing anything with it:

In [ ]:
%%bash
set -euxo pipefail
cat pyproject.toml

### "Or I Could Just Ask uv"

> **Bob:** Wait — I hand-edited a config file like it's 2010. Alice's note says `uv add` does this for me.

`uv add` writes the constraint into `pyproject.toml`, resolves the graph, and installs — all in one step. Add `requests`:

In [ ]:
%%bash
set -euxo pipefail
uv add requests

> **Bob:** It edited my file, locked, *and* installed. Suspiciously convenient.

Removing works the same way. `uv remove` deletes the entry and cleans the package out of the environment:

In [ ]:
%%bash
set -euxo pipefail
uv remove requests

### Dependency Groups

> **Bob:** My test tools don't belong on the production server. There has to be a drawer for those.

There is: `[dependency-groups]`. Add `pytest` to a `dev` group — it lands in the group, not in `dependencies`:

In [ ]:
%%bash
set -euxo pipefail
uv add --group dev pytest

Inspect what `uv` wrote — a new `[dependency-groups]` table, separate from the runtime dependencies:

In [ ]:
%%bash
set -euxo pipefail
cat pyproject.toml

### Dependency Markers

> **Bob:** One of my scripts needs `pywin32` — but only on the Windows box. On this Linux machine that package doesn't even exist.

A PEP 508 marker makes the dependency conditional. Rewrite the file with a platform-guarded `pywin32`:

In [ ]:
%%bash
set -euxo pipefail
cat > pyproject.toml <<'EOF'
[project]
name = "bobs-license-service"
version = "0.1.0"
requires-python = ">=3.11"
dependencies = [
    "fastapi>=0.115,<0.116",
    "httpx==0.24.1",
    "rich>=13.7,<14",
    "pywin32>=310; sys_platform == 'win32'",
]

[dependency-groups]
dev = [
    "pytest>=8.0",
]
EOF

Resolution still succeeds on Linux: the resolver records `pywin32` for Windows environments but skips it here. Without the marker, this command would fail:

In [ ]:
%%bash
set -euxo pipefail
uv lock

## Act III — Lock, Then Sync

### `uv lock` — Resolving the Graph

> **Bob:** So `uv.lock` is the exact bill of materials. My file says "any fastapi 0.115.x" — the lockfile says which one, and every transitive package too.

The lock already happened above; confirm the file exists and see its size:

In [ ]:
%%bash
set -euxo pipefail
ls -la uv.lock

### `uv tree` — What Did I Just Get?

Three declared dependencies pull in a whole graph of transitive ones. Display it:

In [ ]:
%%bash
set -euxo pipefail
uv tree

### `uv sync` — Making the Environment Match

> **Bob:** Lock is truth, `.venv` is reality. Time to make reality match.

`uv sync` installs, upgrades, or removes packages until `.venv` matches `uv.lock` exactly:

In [ ]:
%%bash
set -euxo pipefail
uv sync

## Act IV — Change Is Constant

### Bumping a Pin

> **Bob:** Security bulletin says update `httpx`. Old me would have run `pip install --upgrade` on the server and hoped. New me edits the intent first.

Change the exact pin in `pyproject.toml`:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.24.1/httpx==0.27.2/' pyproject.toml
grep -n 'httpx' pyproject.toml

The file changed, but `uv.lock` still records the old version. Re-resolve so the lockfile reflects the new pin:

In [ ]:
%%bash
set -euxo pipefail
uv lock

Then make the environment follow — lock first, sync second, always in that order:

In [ ]:
%%bash
set -euxo pipefail
uv sync

### A Broken Pin Fails Early

> **Bob:** What happens if I fat-finger a version that doesn't exist? On the old setup I'd find out at 2 a.m. when the service didn't start.

Write an impossible pin:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==0.27.2/httpx==999.0.0/' pyproject.toml
grep -n 'httpx' pyproject.toml

Resolution fails immediately — at the desk, not in production. The `|| true` keeps the notebook running past the expected error:

In [ ]:
%%bash
set -euxo pipefail
uv lock || true

### Repairing the Pin

Restore the valid version:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/httpx==999.0.0/httpx==0.27.2/' pyproject.toml
grep -n 'httpx' pyproject.toml

Re-resolve to confirm the repair:

In [ ]:
%%bash
set -euxo pipefail
uv lock

## Act V — Trust, but Verify

### Drift Detection

> **Bob:** Suppose a colleague edits `pyproject.toml` and forgets to re-lock. Intent and truth drift apart — silently. I want an alarm for that.

Simulate the forgetful colleague — edit the `rich` constraint without re-locking:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.7,<14/rich>=13.8,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

`uv lock --check` compares intent against truth and fails on drift. This is the command for CI and pre-commit:

In [ ]:
%%bash
set -euxo pipefail
uv lock --check || true

Revert the edit:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.8,<14/rich>=13.7,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

Now the check passes again:

In [ ]:
%%bash
set -euxo pipefail
uv lock --check

### Frozen Installs

> **Bob:** In CI I want zero cleverness. Install exactly what was reviewed, or fail loudly.

Create drift again — change `rich` without re-locking:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/rich>=13.7,<14/rich>=13.9,<14/' pyproject.toml
grep -n 'rich' pyproject.toml

`uv sync --frozen` refuses to touch a stale lockfile. A plain `uv sync` would quietly re-lock; `--frozen` fails instead:

In [ ]:
%%bash
set -euxo pipefail
uv sync --frozen || true

Refresh the lockfile the explicit way:

In [ ]:
%%bash
set -euxo pipefail
uv lock

With intent and truth aligned, the frozen install succeeds:

In [ ]:
%%bash
set -euxo pipefail
uv sync --frozen

## Act VI — Upgrades on Bob's Terms

### Exact Pins Don't Move

> **Bob:** Maintenance day. Let's see who decides what gets upgraded — me, or the tool.

Pin `fastapi` to one exact version:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi>=0.115,<0.116/fastapi==0.115.0/' pyproject.toml
grep -n 'fastapi' pyproject.toml

Ask for an upgrade of that one package. Nothing moves — the constraint allows exactly one version, and the resolver respects it:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade-package fastapi

### Ranges Move Within Bounds

Loosen the constraint back to a range:

In [ ]:
%%bash
set -euxo pipefail
sed -i 's/fastapi==0.115.0/fastapi>=0.115,<0.116/' pyproject.toml
grep -n 'fastapi' pyproject.toml

The same command now refreshes `fastapi` to the newest version inside the range — and touches nothing else:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade-package fastapi

### The Maintenance Pass

> **Bob:** And once a month: everything to the newest version my ranges allow, in one reviewable lockfile change.

Upgrade all dependencies within their declared bounds:

In [ ]:
%%bash
set -euxo pipefail
uv lock --upgrade

## Act VII — The Old Reflexes: `uv pip` and Tools

### `uv pip` — pip, but Faster

> **Bob:** Muscle memory is real. My fingers still type `pip install`. Turns out `uv` has a compatibility layer for exactly this reflex.

`uv pip` mirrors classic `pip` commands against the project's `.venv` — fast, but it bypasses `pyproject.toml` and `uv.lock` entirely. Install a stray package:

In [ ]:
%%bash
set -euxo pipefail
uv pip install cowsay

The package is in the environment now — but declared nowhere:

In [ ]:
%%bash
set -euxo pipefail
uv pip list | grep -i cowsay

> **Bob:** And this is the trap, isn't it? The lockfile doesn't know about it.

Correct. The next `uv sync` restores reality to match truth — and removes the stray package:

In [ ]:
%%bash
set -euxo pipefail
uv sync

Verify it is gone. `uv pip` is an escape hatch for experiments, not a workflow:

In [ ]:
%%bash
set -euxo pipefail
uv pip list | grep -i cowsay || echo 'cowsay is gone - the lockfile won'

### `uvx` — Run a Tool Without Installing

> **Bob:** Sometimes I just want to *run* a tool once. Not install it, not declare it — run it.

`uvx` executes a tool in a temporary, cached environment. The project stays untouched:

In [ ]:
%%bash
set -euxo pipefail
uvx cowsay -t 'moo from a temporary environment'

### `uv tool` — Keep a Tool Around

For tools used daily, install them user-wide — permanently on `PATH`, still outside every project:

In [ ]:
%%bash
set -euxo pipefail
uv tool install ruff

List the installed tools:

In [ ]:
%%bash
set -euxo pipefail
uv tool list

Uninstall to leave the machine clean:

In [ ]:
%%bash
set -euxo pipefail
uv tool uninstall ruff

## Epilogue — Bob's Runbook Notes

> **Bob:** New page in the runbook. Alice gets a coffee for the sticky note.

### File, Command, Effect

| File | Changed by | Read by |
| --- | --- | --- |
| `pyproject.toml` | you, `uv add`, `uv remove` | `uv lock` |
| `uv.lock` | `uv lock`, `uv add`, `uv remove` | `uv sync` |
| `.venv` | `uv sync` | your application |

### Bob's Three Rules

1. **Declare intent, don't install by hand.** Dependencies go through `uv add` into `pyproject.toml` — never `pip install` straight into an environment.
2. **Lock before sync, check before commit.** `uv lock` records truth, `uv sync` applies it, `uv lock --check` catches drift.
3. **Frozen in CI.** `uv sync --frozen` guarantees the pipeline installs exactly what was reviewed — a stale lockfile fails the build instead of the deployment.